# Review and validate v2 data
This notebook reads only the frozen v2 files. It checks counts, global prompt separation, IDs, token lengths, provenance, and representative examples.

In [ ]:
import os, sys, json
if os.path.exists('/content'):
    if not os.path.exists('/content/mfr-dpo'):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    REPO = '/content/mfr-dpo'
else:
    REPO = '..'
sys.path.insert(0, f'{REPO}/src')
import mfr_data
splits = mfr_data.load_splits(f'{REPO}/data/v2')
print(json.dumps(json.load(open(f'{REPO}/data/v2/manifest.json')), indent=2)[:4000])


In [ ]:
mfr_data.validate_splits(splits, max_tokens=1024, expected_sizes={'train': 2000, 'val': 200, 'test': 300})
for dataset, parts in splits.items():
    for split, frame in parts.items():
        longest = frame.prompt_tokens + frame[['chosen_tokens','rejected_tokens']].max(axis=1)
        print(f'{dataset:8s} {split:5s}: {len(frame):4d} rows, max {longest.max():4d} tokens')


In [ ]:
for dataset in splits:
    row = splits[dataset]['train'].sample(1, random_state=0).iloc[0]
    print('\n', dataset.upper(), row['id'], row['source_dataset'], row['source_row_id'])
    print('PROMPT:', row.prompt[:500])
    print('CHOSEN:', row.chosen[:500])
    print('REJECTED:', row.rejected[:500])
